# Motor wavelet

**Para que serve:** rodar a transformada wavelet em **uma série pronta** e gerar as figuras.
Tudo que você muda fica na célula de **configuração**; o resto não precisa ser editado.

- **Entrada:** uma série de `02_dados/series_prontas/` (gerada por um notebook de tratamento).
- **Saída:** figuras em `02_dados/resultados/wavelet/<fonte>/<variavel>/<wavelet>_<passo>/` (sobrescreve a execução anterior).

O motor descobre o **passo de tempo** sozinho pelas datas e **para com aviso** se a série tiver datas irregulares ou falhas.
Para outra série (outro passo, variável ou fonte), troque `SERIE` e os textos e rode de novo.

| Onde mexer | Arquivo |
|---|---|
| criar um gráfico novo | `src/wavelet/graficos.py` (função `plotar_<nome>` + `GRAFICOS_DISPONIVEIS`) |
| criar um teste de significância | `src/wavelet/significancia.py` (função `teste_<nome>` + `TESTES_DISPONIVEIS`) |

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RAIZ = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "config").is_dir())
sys.path.insert(0, str(RAIZ))

from config.caminhos_e_parametros import PASTA_SERIES_PRONTAS
from src.wavelet import graficos, motor, significancia

print("Séries prontas:", sorted(p.stem for p in PASTA_SERIES_PRONTAS.glob("*.csv")))
print("Gráficos:", list(graficos.GRAFICOS_DISPONIVEIS))
print("Testes:", list(significancia.TESTES_DISPONIVEIS))

## Configuração

In [ ]:
# --- Série (nome do arquivo em 02_dados/series_prontas, sem .csv) ---
SERIE = "sonda_tp_sfc_1h"

# --- Textos dos títulos ---
FONTE = "SONDA"
LOCAL = "Cachoeira Paulista"
VARIAVEL = "Temperatura do ar"
UNIDADE = "°C"
TEXTO_EXTRA = ""          # aparece na 2ª linha do título (ex.: "2 registros interpolados")
FUSO = "UTC"              # aparece no eixo das datas

# --- Análise ---
WAVELET = "MORLET"        # MORLET, PAUL ou DOG (DOG não tem fase)
TESTES = ["local", "global"]
NIVEL = 0.95
NORMALIZAR = True         # divide pelo desvio padrão (potência em variância normalizada)
PERIODO_MIN = None        # None = automático (2 passos); ou ex.: "1h", "2D"
PERIODO_MAX = None        # None = automático (até o limite do COI); ou ex.: "365D"

# --- Figuras ---
GRAFICOS = ["sinal", "escalograma", "espectro_global", "fase"]
INICIO_DESENHO = None     # recorta só o desenho (ex.: "2025-01-01"); a transformada usa a série inteira
FIM_DESENHO = None
ESMAECER = False          # esmaece as regiões não significativas do escalograma
SALVAR = True

## Cálculo

In [ ]:
serie = motor.carregar_serie_pronta(SERIE)
preparada = motor.preparar_serie(serie, normalizar=NORMALIZAR)
resultado = motor.calcular(preparada, WAVELET, testes=TESTES, nivel=NIVEL,
                           periodo_min=PERIODO_MIN, periodo_max=PERIODO_MAX)
print(motor.resumo(preparada, resultado))

## Figuras

In [ ]:
info = {"fonte": FONTE, "local": LOCAL, "variavel": VARIAVEL, "unidade": UNIDADE,
        "texto_extra": TEXTO_EXTRA, "fuso": FUSO}
opcoes = {"inicio": INICIO_DESENHO, "fim": FIM_DESENHO, "esmaecer": ESMAECER}
salvos = graficos.gerar_graficos(preparada, resultado, info, GRAFICOS, opcoes, salvar=SALVAR)
for caminho in salvos:
    print(caminho)